# Open Design on Jute — M4 Runtime Access Surface Design Spec

> **Status:** draft for review · **Date:** 2026-06-01 · **Depends on:** M3.5 (design-system library, landed) + M2c (deck-theme library, landed).

M4 finalizes the **access surface** for the two already-vendored Open Design libraries so the
`open-design` skill works in a **shipped Jute app**, not just an in-repo dev checkout. This is the
item M3.5 and M2c explicitly deferred: *"runtime install path + MCP surface."*

## Context — what shipped vs. the gap M4 closes

| Milestone | Shipped |
|---|---|
| M1 | `open-design` brain skill (discovery→direction→plan→artifact→critique), notebook-driven via `notebook_*` MCP |
| M2a | Native deck track (`kind: deck`, `deck-mode.md`, 12 layouts, present mode) |
| M2b | 5 OD directions ported into Jute's native deck `THEMES` (OKLch tokens) |
| M2c | **Deck-theme library** — 51 vendored themes + `deck-skeleton.html` under `assets/open-design-deck-library/`, indexed; artifact-deck escalation wired |
| M3.5 | **Design-system library** — 148 `DESIGN.md` under `assets/open-design-library/`, indexed; Direction step wired |

**The gap.** Selection today is `Read`-driven against a **repo-relative path**:
`references/design-systems.md` instructs the brain to `Read assets/open-design-library/index.json`.
That path exists in a source checkout but **does not exist in a packaged Jute.app** — the assets
ship inside the Tauri resource bundle at an OS-specific `resource_dir()`, and the brain has no
portable way to find them. M4 gives the brain a **path-independent** way to discover and fetch
library content, and defines where the libraries live at runtime.

## Scope — what M4 is and isn't

Two earlier framings of "M4" exist; this spec reconciles them:
- M3.5/M2c provenance: *"runtime install path + MCP surface"* (for the libraries that are now shipped).
- Asset-library spec (cell 12): *"Mode-skill library + access surface"* — but the deck-theme half of that
  mode library already landed in M2c, so M4's remaining novel work **is** the access surface + install path.

**In scope**
1. **Runtime location + resolution order** for the two shipped libraries (`design-systems`, `deck-themes`).
2. **MCP surface** — finalize locked-decision #3: `Read` stays the portable default; add the single
   `open_design_search` ranking tool + a fetch tool, both path-independent. (MCP Resources optional.)
3. **Rust loader module** + unit tests; register tools in the existing notebook MCP server.
4. **Skill rewiring** — `references/*.md` + `SKILL.md` call the tools (Read = dev fallback); `spur-core` tests.
5. **Tauri resource bundling** of the two asset dirs.

**Deferred (M4+)**
- Vendoring additional *generic* mode-skill packages beyond deck themes.
- User marketplace / GitHub install into `.spur/open-design/` (port of OD `library-install.ts`).
- Embeddings-based ranking (M4 ranking is deterministic).
- MCP Resources if target clients don't consume them well.

## Verified findings (codebase, with citations)

**F1 — MCP tool registration is plain Rust + flat dispatch.** Each tool = `pub fn tool() -> Tool`
(inline JSON schema) + `pub async fn call(deps: &ServerDeps, args: Value) -> Result<CallToolResult, McpError>`.
Registered in `crates/spur-notebook/src/mcp/tools/mod.rs:43` (`tools()`), dispatched by a literal
`match` arm in `mod.rs` `ServerHandler::call_tool` (~`mod.rs:145`). No macro/derive. New tools follow
this exactly. Representative: `tools/add_api_datasource.rs:13` (params struct) → `:45` (`call`).

**F2 — `ServerDeps` carries the runtime handles** (`mcp/mod.rs:57`): `bridge`, `state`, `daemon`,
and `app: Option<tauri::AppHandle>`. `app.path().resource_dir()` is the bundle-resource locator.

**F3 — Install precedent = copy-on-first-run** (`extension_install.rs:32-56`, called from
`main.rs:353-375`): resolve `resource_dir()` → if dest under `~/.spur/extensions/` missing, `fs::copy`
from `resource_root/extensions/<file>`; leave existing untouched; tolerate missing source. `BaseDirs`
resolves `~/.spur/...` with `$HOME` fallback. **M4's install/resolution mirrors this shape and its tests.**

**F4 — No `include_dir`/`rust-embed`** anywhere; assets are read from disk. So M4 reads files at
runtime from a resolved root (not compiled-in).

**F5 — Asset layout (already committed):**
- `assets/open-design-library/` → `design-systems/<id>/DESIGN.md` ×148 + `index.json` (`kind:"design-systems"`, `items:[{id,title,category,summary,swatches[]}]`).
- `assets/open-design-deck-library/` → `deck-themes/<id>/` (each: `SKILL.md`, `example.html`, `assets/`, `references/`, `LICENSE`) + shared `deck-skeleton.html` + `index.json` (`kind:"deck-themes"`).

**F6 — Skill is prompt-driven.** `SKILL.md` names tool calls in backticks; `references/design-systems.md`
and `references/deck-artifact.md` tell the model to `Read` repo-relative asset paths. No bridging code exists.
There is a tolerated **naming-style split**: skill text uses `notebook_insert_cell` (underscore) while some
registered tools are `notebook.insert_cell` (dot). M4 adopts the **underscore** `open_design_*` family.

## Inherited locked decisions (asset-library spec) and how M4 honors them

| # | Locked decision | M4 obligation |
|---|---|---|
| 2 | On-disk vendored library, **never `include_str!`** | Loader reads from a resolved disk root (F4). |
| 3 | **`Read` default**; MCP **Resources** optional; **one `open_design_search` tool** for ranking only | M4 ships `Read` fallback + `open_design_search` (+ a fetch tool); Resources optional. |
| 4 | Index consulted **on demand at selection**, never at Discovery | Tools are called in Direction/Artifact steps, not Discovery. |
| 6 | Bundled defaults + user adds in `.spur/open-design/` | Resolution overlays user dir over bundle. |

## Runtime location + resolution order

```mermaid
flowchart TD
  call["open_design_* tool called"] --> r{resolve library root for kind}
  r -->|1 env override| env["$SPUR_OPEN_DESIGN_LIBRARY/&lt;kind&gt;\n(dev / tests)"]
  r -->|2 user overlay| usr["~/.spur/open-design/&lt;kind&gt;/\n(user additions, precedence)"]
  r -->|3 bundled| res["resource_dir()/open-design-&lt;lib&gt;/&lt;kind&gt;/\n(shipped default)"]
  r -->|4 dev fallback| repo["crates/spur-notebook/assets/...\n(cargo run from source)"]
  env --> idx[(index.json + package files)]
  usr --> idx
  res --> idx
  repo --> idx
```

- `<kind>` ∈ {`design-systems`, `deck-themes`}; `<lib>` maps kind→dir (`open-design-library`, `open-design-deck-library`).
- **User overlay merges over bundle by `id`** (a user `deck-themes/<id>/` shadows the bundled one) — locked-decision #6.

### Install model: read-in-place, not copy (recommended)
The DuckDB extension is copied (F3) because DuckDB must `LOAD` it from a writable path. The Open Design
libraries are **read-only markdown/JSON** — no copy is required. **Recommendation:** read in place from
`resource_dir()`; create `~/.spur/open-design/<kind>/` only as an *optional, initially-empty overlay* for
user additions. This avoids duplicating ~2 MB on every launch and keeps a single source of truth.
*(Alternative, if a stable user-space path is later required for marketplace installs: seed-copy on first
run exactly like `install_bundled_extension_into`. Deferred to M4+.)*

### Tauri bundling
`tauri.conf.json` currently has **no `bundle.resources`** key. M4 adds the two asset dirs so they land
under `resource_dir()`:
```json
"bundle": {
  "resources": {
    "assets/open-design-library": "open-design-library",
    "assets/open-design-deck-library": "open-design-deck-library"
  }
}
```
*(The existing DuckDB `extensions/` mapping, wherever configured, stays unchanged.)*

## MCP surface — `open_design_*` tool family

Added to the **existing notebook MCP server** (new tool group; underscore naming per F6). Three tools:

### 1. `open_design_search` — the one ranking action (locked-decision #3)
```jsonc
// params
{ "query": "string (brand / vibe / brief keywords)",
  "kind":  "design-systems" | "deck-themes" | null,   // null = search both
  "limit": 8 }
// returns
{ "items": [ { "id", "kind", "title", "category", "summary", "swatches": ["#hex"], "score": 0.0 } ] }
```
Server-side deterministic ranking over the merged index (see Ranking). This is the only tool that
*computes a decision*; the rest just fetch.

### 2. `open_design_get` — path-independent package fetch
```jsonc
// params
{ "kind": "design-systems" | "deck-themes", "id": "string",
  "include_skeleton": false }   // deck-themes only: also return shared deck-skeleton.html
// returns (design-systems)
{ "id", "kind":"design-systems", "design_md": "…full DESIGN.md…" }
// returns (deck-themes)
{ "id", "kind":"deck-themes", "skill_md": "…", "example_html": "…",
  "deck_skeleton_html": "… (when include_skeleton) …",
  "files": [ { "path": "assets/foo.css", "bytes": 1234 } ] }   // manifest of side files, fetch-on-demand
```
Solves the multi-file bundle problem: deck themes are `SKILL.md` + `example.html` + `assets/`; the brain
gets them atomically without knowing the resolved path. (Large binary side files are listed, not inlined.)

### 3. `open_design_list` — cheap enumerate (optional)
```jsonc
{ "kind": "design-systems" | "deck-themes" } -> { "count", "items": [ index.json rows ] }
```
Thin wrapper over the resolved `index.json` for when the brain wants the whole menu. May be folded into
`open_design_search` with an empty query.

### MCP Resources (optional, defer unless trivial)
`opendesign://design-systems/{id}`, `opendesign://deck-themes/{id}` — idiomatic host-managed loading for
clients that consume Resources. Gate on client support; `open_design_get` covers the same need portably.

## Ranking (deterministic, no embeddings)

`open_design_search` ranks with a transparent field-weighted token score — testable, no model calls:
- Tokenize `query` (lowercase, split on non-alphanumerics).
- Per item, sum weighted matches: `id`/`title` exact-token = 3, `category` = 2, `title`/`summary`
  substring = 1; **color queries** (a `#hex` or a known color word) match against `swatches`.
- Tie-break by `id` for stable ordering; return top `limit`.

Rationale: the corpus is ~200 items — substring/field weighting is "fine for dozens/hundreds" (asset-library
spec, open decision #4). Embeddings are an M4+ upgrade behind the same tool signature.

## Rust implementation surface

| File | Change | Responsibility |
|---|---|---|
| `crates/spur-notebook/src/open_design/mod.rs` | **new** | module root; `Kind` enum; `LibraryError` |
| `crates/spur-notebook/src/open_design/library.rs` | **new** | `resolve_root(kind, deps) -> PathBuf` (env→user→resource→repo), `load_index`, `get_design_system`, `get_deck_theme`, `search` — pure, unit-tested w/ tempdirs (mirror `extension_install` tests) |
| `crates/spur-notebook/src/mcp/tools/open_design_search.rs` | **new** | `tool()`+`call()` → `library::search` |
| `crates/spur-notebook/src/mcp/tools/open_design_get.rs` | **new** | `tool()`+`call()` → `library::get_*` |
| `crates/spur-notebook/src/mcp/tools/open_design_list.rs` | **new** (optional) | `tool()`+`call()` → `library::load_index` |
| `crates/spur-notebook/src/mcp/tools/mod.rs` | **modify** | add the new `tool()`s to `tools()` |
| `crates/spur-notebook/src/mcp/mod.rs` | **modify** | add `match` arms in `ServerHandler::call_tool` |
| `crates/spur-notebook/src/lib.rs` | **modify** | `pub mod open_design;` |
| `crates/spur-notebook/tauri.conf.json` | **modify** | add `bundle.resources` (two dirs) |

`resolve_root` reads `resource_dir()` via `deps.app` (F2); when `app`/resource is absent (headless tests,
`cargo run`), it falls through to `$SPUR_OPEN_DESIGN_LIBRARY` then the repo-relative `assets/` path so dev
and CI work without a bundle.

## Skill rewiring

- `crates/spur-core/src/skills/open-design/references/design-systems.md` — replace the raw-`Read`
  instruction with: *call `open_design_search({query, kind:"design-systems"})`, then `open_design_get`
  the chosen `id`*; keep `Read` of the repo path as an explicit **dev fallback** note.
- `crates/spur-core/src/skills/open-design/references/deck-artifact.md` — same pattern for
  `kind:"deck-themes"`; fetch `deck_skeleton_html` via `open_design_get(..., include_skeleton:true)`.
- `crates/spur-core/src/skills/open-design/SKILL.md` — add `open_design_search` / `open_design_get`
  to the tool roster in the `<HARD-GATE>`.
- `crates/spur-core/src/skills/mod.rs` — extend the existing assertions: the two references must mention
  `open_design_search` and `open_design_get` (mirrors the current `references/...` / `index.json` checks).

## UI / UX mockup — selection flow (rendered as a `text/html` cell output)

The brain's experience in the Direction step: one `open_design_search` call returns a ranked menu; one
`open_design_get` binds the choice. The mockup below renders that flow (run the cell; Jute shows the HTML).

In [1]:
# M4 selection-flow mockup — search -> ranked menu -> get -> bind. Renders as text/html.
from IPython.display import HTML
rows = [
    ("stripe",   "design-systems", "Stripe",        "Fintech",   ["#635bff","#0a2540"], 0.92),
    ("linear",   "design-systems", "Linear",        "Productivity",["#5e6ad2","#08090a"], 0.71),
    ("guizang-ppt","deck-themes",  "Guizang PPT",   "Editorial", ["#111111","#e8e2d6"], 0.55),
]
def chips(cs): return "".join(f'<span style="display:inline-block;width:14px;height:14px;border-radius:3px;background:{c};margin-right:3px;vertical-align:middle"></span>' for c in cs)
trs = "".join(f"""<tr>
  <td style="font-family:ui-monospace,monospace;color:#5e6ad2">{r[0]}</td>
  <td style="color:#6b7280">{r[1]}</td><td>{r[2]}</td><td style="color:#6b7280">{r[3]}</td>
  <td>{chips(r[4])}</td><td style="text-align:right;font-variant-numeric:tabular-nums">{r[5]:.2f}</td></tr>""" for r in rows)
HTML(f"""
<div style="font-family:-apple-system,Inter,sans-serif;max-width:680px;border:1px solid #e5e7eb;border-radius:12px;overflow:hidden">
  <div style="padding:12px 16px;background:#0a2540;color:#fff;font-size:13px">
    <code style="color:#9ad">open_design_search</code>({{ query:"fintech, trustworthy, purple", kind:null, limit:8 }})</div>
  <table style="width:100%;border-collapse:collapse;font-size:13px">
    <thead><tr style="text-align:left;color:#9ca3af;font-size:11px;text-transform:uppercase">
      <th style="padding:8px 16px">id</th><th>kind</th><th>title</th><th>category</th><th>swatches</th><th style="text-align:right;padding-right:16px">score</th></tr></thead>
    <tbody>{trs}</tbody></table>
  <div style="padding:10px 16px;background:#f9fafb;color:#374151;font-size:12px;border-top:1px solid #eee">
    &rarr; <code style="color:#635bff">open_design_get</code>({{ kind:"design-systems", id:"stripe" }}) &rarr; bind palette to artifact <code>:root</code></div>
</div>""")

id,kind,title,category,swatches,score
stripe,design-systems,Stripe,Fintech,,0.92
linear,design-systems,Linear,Productivity,,0.71
guizang-ppt,deck-themes,Guizang PPT,Editorial,,0.55


## Token-budget / progressive-disclosure check

- **Discovery** loads nothing from the library (locked-decision #4).
- **Direction**: one `open_design_search` → a ranked **menu** (~8 rows, id/title/category/swatches/summary)
  — small, bounded. The full `DESIGN.md` / `SKILL.md` is fetched only **after** a pick (`open_design_get`).
- This is strictly cheaper than the brain `Read`-ing `index.json` wholesale (148+51 rows), and it removes
  the need to hardcode any path. Net: fewer tokens, path-independent, one decision tool.

## Milestones / task breakdown (for the plan)

1. **Loader module** — `open_design/{mod,library}.rs`: resolution order + `load_index`/`get_*`/`search`,
   with tempdir unit tests (env override, user-overlay precedence, missing-bundle dev fallback, `bmw-m`
   empty-swatch edge case). *No MCP yet.*
2. **MCP tools** — `open_design_search` + `open_design_get` (+ optional `_list`); register in
   `tools/mod.rs` + `mcp/mod.rs`; structured-output returns. Tool-schema test.
3. **Tauri bundling** — add `bundle.resources`; verify `resource_dir()` resolution path in a guarded test.
4. **Skill rewiring** — update `design-systems.md` + `deck-artifact.md` + `SKILL.md`; extend `spur-core`
   skills tests; full `cargo test -p spur-core --lib skills` green.
5. **Provenance** — CREATION-LOG M4 entry; this spec referenced.

Each task is independent-ish and file-scoped for parallel workers (loader before tools; tools before
skill rewiring references the tool names).

## Open decisions to settle before the plan

1. **Install model:** read-in-place from `resource_dir()` (recommended) vs seed-copy to `~/.spur/open-design/`.
2. **Tool set:** ship `open_design_search` + `open_design_get` only, or also `open_design_list`?
3. **MCP Resources:** include `opendesign://…` now, or defer to M4+? (Lean: defer.)
4. **Server placement:** add `open_design_*` to the **notebook** MCP server (recommended — one socket,
   shares `ServerDeps`/`resource_dir`) vs a separate Open Design MCP server.
5. **Naming:** confirm `open_design_*` (underscore) + `opendesign://` scheme.
6. **Ranking:** deterministic field-weighted (recommended) — confirm embeddings stay M4+.

**Next:** on approval, turn this into a `submit_plan` (tasks above).